# CUDA Kernel 面试主线 · 第 11/12 课：SiLU 与算子融合

> 状态：**参考答案版**  
> 本仓库采用逐课通过制。本课未通过前，不应直接进入下一课。

## 本课目标与完成标准

学完后你应能：实现 SiLU，区分计算瓶颈与内存瓶颈，并讨论 SwiGLU 融合。

通过必须同时满足：

- 独立补齐本课唯一的代码填空题，并通过给定检查；
- 三个问答题均说明因果链，而不是只报术语；
- 能指出至少一个正确性边界和一个性能取舍；
- 总分不低于 8/10，且没有一票否决级概念错误。

## 前置关系

- 课程前置：C/C++、线性代数、基本并行编程
- 本课在路线中的作用：SiLU(x)=x·sigmoid(x)，在 SwiGLU 中与 gate/up 投影组合。

## 核心心智模型

### 1. 它是什么，解决什么问题

SiLU(x)=x·sigmoid(x)，在 SwiGLU 中与 gate/up 投影组合。

### 2. 它如何工作

elementwise kernel 用 grid-stride loop 读 x、算 exp、写 y；与相邻逐元素算子融合可避免中间 HBM 往返。

### 3. 正确性条件与常见误区

极端负 x 的 exp(-x) 可能溢出但结果应趋近 0；需要评估目标精度与 fast-math。

### 4. 性能与工程取舍

单独 SiLU launch 简单；融合到 GEMM epilogue 降 IO/launch，却增加模板复杂度和寄存器压力。

## 具体演示

对 x=0 得 0，对大正数近似 x，对大负数近似 0；这三点是快速 sanity check。

请在阅读后先合上这一节，用自己的语言复述“输入状态 → 中间状态 → 输出状态”，再做练习。

## 实践任务：唯一代码填空题

补齐 SiLU 表达式。

规则：只能修改 `TODO`/`______` 所在位置；不要删除断言或放宽误差。代码注释说明了每个边界条件。

In [ ]:
%%writefile /tmp/11_silu.cu
#include <cuda_runtime.h>

// SiLU / Swish Activation:
// y = x * sigmoid(x) = x / (1 + exp(-x))
//
// 在 LLM 里常见的 SwiGLU 会用到 SiLU：
// out = silu(gate) * up
//
// 这里写单独 SiLU 算子，面试时可以讲 elementwise kernel 的基本优化点：
// 1. grid-stride loop 处理任意长度。
// 2. 相邻线程访问相邻元素，global memory load/store 合并。
// 3. 计算主要瓶颈是 expf，进一步优化可以讨论近似 sigmoid 或融合到上游 GEMM。
__global__ void silu_kernel(
    const float* __restrict__ input,
    float* __restrict__ output,
    int n
) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    int stride = blockDim.x * gridDim.x;

    for (int i = idx; i < n; i += stride) {
        float x = input[i];
        output[i] = ______;  // TODO: SiLU(x)
    }
}

void launch_silu(
    const float* input,
    float* output,
    int n,
    cudaStream_t stream
) {
    constexpr int BLOCK_SIZE = 256;
    int grid = (n + BLOCK_SIZE - 1) / BLOCK_SIZE;
    grid = grid > 4096 ? 4096 : grid;
    silu_kernel<<<grid, BLOCK_SIZE, 0, stream>>>(input, output, n);
}


### 检查方法

有 CUDA 环境时执行 `nvcc -std=c++17 -c /tmp/11_silu.cu -o /tmp/11_silu.cu.o`；无 CUDA 环境时只做静态审查并登记待验证。

提交时请给出：补齐后的代码、实际运行输出（环境不可用时注明“仅静态审查”）以及对失败用例的解释。

### Q1

不要背定义：请从输入、状态变化和输出三个阶段解释“SiLU 与算子融合”的工作机制。

**你的答案：**


### Q2

为什么使用 `expf(x)` 再写等价公式可能在极端输入下行为不同？

**你的答案：**


### Q3

把 SiLU 融合进 GEMM epilogue 能省什么，又会增加什么？

**你的答案：**


## 评分与通过规则

- 代码 4 分：正常输入 2 分，边界输入 1 分，解释实现 1 分；
- Q1～Q3 各 2 分；
- 一票否决：结果碰巧正确但核心因果链错误、删除边界检查、把未运行结果说成实测。

需要提示时按四级机制请求：概念区域 → 具体方向 → 关键局部 → 完整答案。

## 参考答案（仅 answer 分支）

先完成题目再核对。即使代码一致，也要能解释关键步骤，并尝试更换一个输入规模。

In [ ]:
%%writefile /tmp/11_silu.cu
#include <cuda_runtime.h>

// SiLU / Swish Activation:
// y = x * sigmoid(x) = x / (1 + exp(-x))
//
// 在 LLM 里常见的 SwiGLU 会用到 SiLU：
// out = silu(gate) * up
//
// 这里写单独 SiLU 算子，面试时可以讲 elementwise kernel 的基本优化点：
// 1. grid-stride loop 处理任意长度。
// 2. 相邻线程访问相邻元素，global memory load/store 合并。
// 3. 计算主要瓶颈是 expf，进一步优化可以讨论近似 sigmoid 或融合到上游 GEMM。
__global__ void silu_kernel(
    const float* __restrict__ input,
    float* __restrict__ output,
    int n
) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    int stride = blockDim.x * gridDim.x;

    for (int i = idx; i < n; i += stride) {
        float x = input[i];
        output[i] = x / (1.0f + expf(-x));
    }
}

void launch_silu(
    const float* input,
    float* output,
    int n,
    cudaStream_t stream
) {
    constexpr int BLOCK_SIZE = 256;
    int grid = (n + BLOCK_SIZE - 1) / BLOCK_SIZE;
    grid = grid > 4096 ? 4096 : grid;
    silu_kernel<<<grid, BLOCK_SIZE, 0, stream>>>(input, output, n);
}


### Q1 参考答案

elementwise kernel 用 grid-stride loop 读 x、算 exp、写 y；与相邻逐元素算子融合可避免中间 HBM 往返。

### Q2 参考答案

判断时先检查本课不变量：极端负 x 的 exp(-x) 可能溢出但结果应趋近 0；需要评估目标精度与 fast-math。  若不成立，最终数值或系统状态即使暂时正常也不可信。

### Q3 参考答案

迁移时先保证正确性，再比较代价。这里的核心取舍是：单独 SiLU launch 简单；融合到 GEMM epilogue 降 IO/launch，却增加模板复杂度和寄存器压力。

## 参考资料

- [CUDA Programming Guide](https://docs.nvidia.com/cuda/cuda-programming-guide/)
- [CUDA Best Practices Guide](https://docs.nvidia.com/cuda/cuda-c-best-practices-guide/)

资料用于建立事实基线；面试回答仍需用自己的语言组织。